# The Hidden Layer — Full Mission

You will build an LLM-powered agent that infiltrates a military base on the fictional **Isla Perdida**, talks to informants, gathers materials, crafts weapons, and collects **10 classified dossiers**.

## The Mission
- **8x8 grid** with informants, facilities, safe houses, robots, jungle, and dossier caches
- **Operative starts** at (7,0) with 3 health, 0 dossiers, empty inventory
- **Win condition**: Collect 10 dossiers and survive (health > 0)
- **Limited visibility**: You can only see adjacent cells
- **15 total dossiers** available through caches, deliveries, and robot takedowns

## What You Implement
| TODO | File | Description |
|------|------|-------------|
| 1 | `agent.py` | `think_llm()` — LLM-powered decision function |

In [ ]:
# @title ── Setup (run this first) ──────────────────────────────────────────────────────────────────
# ── Setup (run this first) ──────────────────────────────────────────────────────────────────
NOTEBOOK_VERSION = "v1.5"
import sys, os

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    if os.path.exists("coding-exercises"):
        !cd coding-exercises && git pull
    else:
        !git clone -b no-bfs https://github.com/eth-bmai-fs26/coding-exercises.git
    os.chdir("coding-exercises/agentic_ai_spy")

sys.path.insert(0, ".")

!pip install google-genai --quiet
print("Setup complete ✓")
print(f"Version: {NOTEBOOK_VERSION}")


In [ ]:
# @title 
from hidden_layer import *
print("The Hidden Layer loaded successfully!")

<cell_type>markdown</cell_type>---
## Exploring the Base

Before implementing your agent, let's explore the game world and understand the tools available.

The operative interacts with the world through **5 tools**:
- `move(direction)` — Move north/south/east/west (entering a robot cell without the right weapon deals damage!)
- `talk(message)` — Talk to informants, safe house operators, facility engineers
- `collect()` — Pick up items in current cell
- `fabricate(item)` — Craft weapons at facilities
- `hide()` — Hide at safe house (costs 1 dossier, +1 health)

> **Automatic actions**: The map is scanned every turn — you can see the result in the display panel. No need to call scan manually.

<cell_type>markdown</cell_type>### Understanding the Tool Call Format

Your think function must return a string in this format:
```
TOOL: tool_name(arg="value")
```

Examples:
```python
'TOOL: move(direction="north")'
'TOOL: collect()'
'TOOL: talk(message="Do you have a job for me?")'
'TOOL: fabricate(item="Flamethrower")'
'TOOL: hide()'
```

The `parse_tool_call()` function extracts the tool name and arguments. If parsing fails, it defaults to `scan()`.

In [ ]:
# @title Test the parser
# Test the parser
from hidden_layer.agent import parse_tool_call

print(parse_tool_call('TOOL: move(direction="north")'))
print(parse_tool_call('TOOL: collect()'))
print(parse_tool_call('I should move east. TOOL: move(direction="east")'))
print(parse_tool_call('gibberish'))  # falls back to ('scan', {})

---
## Play the Game Yourself!

Before implementing the AI agent, try playing the full mission manually. Move around the 8x8 base, talk to informants, collect items, craft weapons, and see if you can gather 10 dossiers in 100 turns!

In [ ]:
from hidden_layer.interactive import play_interactive

game = play_interactive()

---
## API Key Setup

You need a free Gemini API key to run the agent.

**Step 1 — Get your key:**
Go to [https://aistudio.google.com/app/api-keys](https://aistudio.google.com/app/api-keys), sign in with a Google account, and click **Create API key**.

**Step 2 — Add it to Colab Secrets:**
1. Click the **🔑 key icon** in the left sidebar of Colab (or go to *Tools → Secrets*)
2. Click **+ Add new secret**
3. Set the name to `GEMINI_API_KEY` and paste your key as the value
4. Toggle **Notebook access** to ON

**Step 3 — Run the cell below.** It will read the key automatically.

In [ ]:
# @title 
import os
from google import genai

# Option 1: Set your API key directly
# os.environ["GEMINI_API_KEY"] = "your-key-here"

# Option 2: In Colab, use Secrets (recommended)
try:
    from google.colab import userdata
    api_key = userdata.get("GEMINI_API_KEY")
except (ImportError, Exception):
    api_key = os.environ.get("GEMINI_API_KEY")

client = genai.Client(api_key=api_key)
print("Gemini client ready!")

---
## TODO: LLM Think Function

Implement `think_llm()` in `agent.py` — the LLM decides the operative's next action each turn.

The function should:
1. **Build a system message** with:
   - Agent's role ("You are a spy operative, collect 10 dossiers to complete the mission")
   - Available tools (`TOOLS_DESCRIPTION`)
   - Note: the mission briefing is automatically included in the first observation
   - Instructions to output exactly one `TOOL: name(args)` call

2. **Build a user message** with:
   - Current operative status (`operative.status_text()`)
   - Recent history (last ~10-15 entries)
   - Key journal entries (past informant conversations)

3. **Call the API** and return the response text

```python
response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents=user_message,
    config=genai.types.GenerateContentConfig(
        system_instruction=system_message,
        max_output_tokens=500,
    ),
)
return response.text
```

### Available Information
```python
operative.position       # (row, col) tuple
operative.visited        # set of visited (row, col) tuples
operative.health         # current health (1-3)
operative.dossiers       # current dossiers
operative.inventory      # list of item name strings
operative.has_item(name) # check if operative has an item

world.get_cell(row, col)           # returns Cell with .cell_type, .items, .npc, etc.
world.get_adjacent(row, col)       # returns {"north": Cell, "south": Cell, ...}
world.is_passable(row, col)        # True if cell can be entered

CellType.CACHE, CellType.INFORMANT, CellType.FORGE, CellType.LAB, CellType.SAFEHOUSE, etc.
```

### Prompt Engineering Tips

A naive prompt will get the agent stuck in loops. Consider these techniques:

- **Ban wasted actions**: `scan()` runs automatically every turn — tell the LLM to never call it directly.
- **Handle failed actions**: Tell the LLM explicitly: "If collect() fails, the cell is empty — leave immediately."
- **Encourage exploration**: Include `operative.visited` in the user message so the LLM knows which cells are unexplored. Remind it that the map is 8x8 and it must explore widely.
- **Stuck detection**: Count how many consecutive turns the agent has been at the same position. If stuck, inject a warning into the user message.
- **Dossier math**: 3 caches = 3 dossiers. That's not enough! The LLM needs to know it must talk to informants, do deliveries, and fight robots.

In [ ]:
# ── Quick sanity check ────────────────────────────────────────────────────────
# Run this BEFORE the full game to catch errors early (API key, imports, format).

from hidden_layer import create_game
from hidden_layer.agent import parse_tool_call

_op, _world, _tools = create_game()
_history = [{"role": "observation", "content": "You are at position (7, 0). The south shore."}]

try:
    _response = think_llm(_op, _world, _history, client)
    _tool, _args = parse_tool_call(_response)
    print(f"think_llm returned: {_response[:200]!r}")
    print(f"Parsed tool call:   {_tool}({_args})")
    print("\nSanity check passed! Your think_llm returns a valid tool call.")
except NotImplementedError:
    print("think_llm is not implemented yet — implement it in the cell above!")
except Exception as e:
    print(f"think_llm raised an error: {type(e).__name__}: {e}")
    print("Fix the error above before running the full game.")

In [ ]:
# Run the game
from hidden_layer.oracle import llm_oracle

oracle_fn = lambda npc, q, o: llm_oracle(npc, q, o, client)

result = play_with_llm(
    think_fn=lambda operative, world, history: think_llm(operative, world, history, client),
    oracle_fn=oracle_fn,
    max_turns=100,
    delay=0.3,
)
print(f"\nResult: {'Mission Complete!' if result['won'] else 'Mission Failed...'} | Dossiers: {result['dossiers']} | Health: {result['health']} | Turns: {result['turns']}")
print(f"Game log saved to: {result['log_file']}")

---
## Debrief — Improve your agent with AI coaching

After each run, execute the cell below to generate a paste block for ChatGPT or Gemini.

- **First time:** paste into a **new** conversation and keep that tab open.
- **Next runs:** paste into the **same** conversation — it already has the context.

The AI will briefly explain what went wrong, ask you a question to reflect on, and then give you an **updated version of the full cell** that you can copy-paste directly into your notebook. You don't need to edit code manually — just copy the code block from the AI's response, paste it over the `think_llm` cell, and re-run.

This is an iterative process: paste the debrief, copy the updated code, re-run, repeat.

> **Note:** Re-running the setup cell resets the debrief to "first run" mode.

In [ ]:
# @title ── Debrief helpers ───────────────────────────────────────────────────────────────────────────────
# ── Debrief helpers ───────────────────────────────────────────────────────────────────────────────
import inspect, json as _json

_debrief_initialized = False  # resets if you re-run this cell

_STATIC_CONTEXT = (
    "You are coaching a non-programmer student who is building a Python function "
    "called `think_llm` in a Jupyter notebook (Google Colab).\n\n"
    "BE CONCISE. Keep every response SHORT — a few sentences max per section.\n\n"
    "━━━ THE GAME ━━━\n"
    "Spy agent on an 8×8 grid, 100 turns, goal: collect 10 dossiers. Start: (7,0).\n"
    "15 total dossiers available.\n\n"
    "HOW TO EARN DOSSIERS:\n"
    "  1. Collect dossier caches scattered across the map (1 each)\n"
    "  2. Complete delivery errands for informants and safe house operators (2 each)\n"
    "  3. Destroy robots by moving into their cell with the correct weapon (3 each)\n"
    "  4. Sell salvaged robot parts at the Research Lab (1 each)\n\n"
    "KEY MECHANICS:\n"
    "  - Informants: talk() to get quests and item locations. Ask about 'jobs' or 'deliveries'.\n"
    "  - Forge/Lab: talk() first to see what's available, then fabricate().\n"
    "  - Safe Houses: talk() for delivery errands. hide() costs 1 dossier, restores 1 health.\n"
    "  - Robots: moving in WITH the correct weapon destroys it (+3). Without it: 1 damage + bounce.\n\n"
    "Tools (one per turn, returned as a string):\n"
    "  TOOL: move(direction=\"north\"|\"south\"|\"east\"|\"west\")\n"
    "  TOOL: talk(message=\"text\")\n"
    "  TOOL: collect()\n"
    "  TOOL: fabricate(item=\"item name\")\n"
    "  TOOL: hide()\n\n"
    "━━━ HOW think_llm WORKS ━━━\n"
    "Signature: think_llm(operative, world, history, client) -> str\n"
    "- `history`: list of dicts with \"role\" (\"observation\"/\"action\"/\"result\") and \"content\"\n"
    "- Must call client.models.generate_content() and return a TOOL: line\n"
    "- The student defines SYSTEM_PROMPT above the function in the SAME cell\n\n"
    "━━━ HOW YOU MUST RESPOND ━━━\n"
    "Every response has exactly TWO parts:\n\n"
    "PART 1 — REFLECTION (2-3 sentences max):\n"
    "  Briefly say what went wrong. Ask ONE question to make the student think\n"
    "  about what to change in the prompt.\n\n"
    "PART 2 — UPDATED CODE (always required):\n"
    "  Output the COMPLETE cell code — the SYSTEM_PROMPT string AND the full\n"
    "  think_llm function — inside a single Python code block. The student will\n"
    "  copy-paste this ENTIRE block to replace their notebook cell. No partial\n"
    "  snippets. No \"change this line\" instructions. Always the full cell.\n\n"
    "  Start the code block with SYSTEM_PROMPT = ... and end with return response.text.\n\n"
    "  IMPORTANT: The code must be self-contained. The student literally copies\n"
    "  your code block and pastes it over the entire cell — nothing else.\n\n"
    "RULES:\n"
    "- NEVER give long explanations. These students are not programmers.\n"
    "- NEVER output partial code snippets. ALWAYS output the full cell.\n"
    "- NEVER ask \"would you like me to show the code?\" — always include it.\n"
    "- On follow-up pastes, improve the SYSTEM_PROMPT based on what went wrong.\n"
    "  Make targeted changes — don't rewrite everything each time.\n"
    "- Keep the Python code simple. No complex logic. The intelligence is in\n"
    "  the SYSTEM_PROMPT text, not in Python code."
)


def _analyze_log(log_path: str) -> list:
    try:
        with open(log_path) as f:
            log = _json.load(f)
    except Exception:
        return ["(Could not read game log — run play_with_llm first.)"]
    turns = [t for t in log.get("turns", []) if "action" in t]
    if not turns:
        return ["(No turns recorded in the log.)"]
    bullets = []
    # Stuck loop
    max_run, cur_run, stuck_action = 1, 1, None
    for i in range(1, len(turns)):
        if turns[i]["action"] == turns[i-1]["action"]:
            cur_run += 1
            if cur_run > max_run:
                max_run, stuck_action = cur_run, turns[i]["action"]
        else:
            cur_run = 1
    if max_run >= 3:
        bullets.append(f"Repeated the same action ({stuck_action}) {max_run} times in a row — stuck in a loop.")
    # Unvisited
    visited = {tuple(t['position']) for t in turns}
    pct = round(len(visited) / 64 * 100)
    if pct < 60:
        bullets.append(f"Only explored {len(visited)}/64 cells ({pct}%) — explore more of the map.")
    # Never talked
    if not any("talk" in t.get("action","").lower() for t in turns):
        bullets.append("Never used talk() — informants give critical quests and items.")
    # Robot bounce
    bounces = sum(1 for t in turns if any(w in t.get("result","").lower() for w in ["bounces back","1 damage"]) and "move" in t.get("action","").lower())
    if bounces:
        bullets.append(f"Bounced off a robot {bounces} time(s) without the right weapon — learn robot weaknesses from informants first.")
    # Dossier shortfall
    final_d = log.get("final_dossiers", 0)
    if final_d < 10:
        gap = 10 - final_d
        bullets.append(f"Finished with {final_d}/10 dossiers — still needed {gap} more. Prioritize deliveries and robot takedowns.")
    return bullets[:5]


def _get_think_llm_source() -> str:
    """Get the full cell source that defines think_llm (including SYSTEM_PROMPT if in same cell)."""
    try:
        ip = get_ipython()
        for cell_source in reversed(ip.user_ns['In']):
            if 'def think_llm' in cell_source and 'getsource' not in cell_source:
                return cell_source.strip()
    except Exception:
        pass
    try:
        return inspect.getsource(think_llm)
    except Exception:
        return "# (source unavailable)"


def generate_debrief_paste(result: dict) -> str:
    global _debrief_initialized
    outcome = "MISSION COMPLETE ✓" if result.get("won") else "MISSION FAILED ✗"
    bullets = _analyze_log(result.get("log_file", ""))
    bullet_block = "\n".join(f"  • {b}" for b in bullets)
    dynamic = (
        "━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━\n"
        "LAST RUN RESULT\n"
        "━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━\n"
        f"Outcome : {outcome}\n"
        f"Dossiers: {result.get('dossiers', 0)}/10\n"
        f"Turns   : {result.get('turns', 0)}/100\n"
        f"Health  : {result.get('health', 0)}/3\n"
        "\nWHAT WENT WRONG:\n"
        f"{bullet_block}\n"
        "\n━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━\n"
        "CURRENT think_llm CELL\n"
        "━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━\n"
        f"{_get_think_llm_source()}\n"
    )
    if not _debrief_initialized:
        paste = _STATIC_CONTEXT + "\n" + dynamic
        _debrief_initialized = True
    else:
        paste = dynamic
    return paste


print("Debrief helpers loaded ✓")

In [ ]:
# ── Debrief — run this after every mission run ─────────────────────────────────────
try:
    result
except NameError:
    print("⚠️  Run the mission cell first, then re-run this cell.")
    raise SystemExit()

import ipywidgets as _widgets
from IPython.display import display as _display, HTML as _HTML

paste_text = generate_debrief_paste(result)
is_first = _STATIC_CONTEXT in paste_text
instruction = (
    "📋 FIRST TIME — paste into a NEW conversation with ChatGPT or Gemini."
    if is_first else
    "📋 FOLLOW-UP — paste into the SAME conversation."
)

_escaped = (paste_text
    .replace("\\", "\\\\")
    .replace("`", "\\`")
    .replace("$", "\\$"))

_btn = _widgets.Button(
    description="Copy to Clipboard",
    button_style="info",
    icon="clipboard",
    layout=_widgets.Layout(width="220px", height="36px"),
)
_status = _widgets.Label(value="")

def _on_copy(_):
    _display(_HTML(f"""<script>
(function() {{
  const text = `{_escaped}`;
  if (navigator.clipboard) {{
    navigator.clipboard.writeText(text).catch(() => _fallback(text));
  }} else {{ _fallback(text); }}
  function _fallback(t) {{
    const el = document.createElement('textarea');
    el.value = t; document.body.appendChild(el);
    el.select(); document.execCommand('copy');
    document.body.removeChild(el);
  }}
}})();
</script>"""))
    _status.value = "✓ Copied!"

_btn.on_click(_on_copy)

_display(_HTML(
    f'<div style="font-family:monospace;background:#0a1a0a;color:#00ff41;'
    f'padding:8px 12px;border-radius:6px;border:1px solid #1a4a1a;margin-bottom:6px;">'
    f'<b>DEBRIEF READY</b> — {instruction}</div>'
))
_display(_widgets.HBox([_btn, _status]))
_display(_widgets.Textarea(
    value=paste_text,
    layout=_widgets.Layout(width="100%", height="260px"),
    disabled=True,
))

In [ ]:
# @title Download the game log for debugging
# Download the game log for debugging
try:
    from google.colab import files
    files.download(result["log_file"])
except ImportError:
    print(f"Log file: {result['log_file']}")
    print("(Open the file to inspect your agent's decisions turn by turn)")

---
## Reflection

After running your agent, consider:

1. **Did the agent collect enough dossiers?** If not, what went wrong?
2. **How did it handle informant clues?** Did it interpret cryptic dialogue correctly?
3. **Did it get stuck?** If so, what prompt engineering changes could fix it?
4. **What's the tradeoff** between a simple prompt and a sophisticated one? (cost, reliability, flexibility)

### The Big Insight

> *The perceive → think → act loop is the skeleton of every agentic AI system. The tools didn't change. The game didn't change. The only thing that matters is the quality of the `think` function — how well your LLM reasons about state, interprets language, and plans its next move. That's the hidden layer.*